# DEE — Tensión topológica del vacío T³ vs R³

**Pregunta central:** ¿el cristal cerrado sobre T³ tiene una contribución residual de energía respecto al cristal abierto R³, y esa contribución se comporta como Λ (constante en V) o como Casimir (∝ 1/V)?

**Hipótesis (Bruschi):** la topología cerrada de T³ impone tensiones que no se relajan, generando una densidad residual ρ_top con presión negativa que podría ser candidato a Λ.

**Predicciones competitivas:**

| Escalamiento | Interpretación | ¿Es Λ? |
|---|---|---|
| ΔE ∝ V | ρ_top constante | **SÍ** ✓ |
| ΔE ∝ V^(2/3) | Solo efecto borde | NO |
| ΔE ∝ V^(1/3) | Defectos lineales | NO |
| ΔE ∝ 1/V | Casimir clásico | NO (se diluye) |
| ΔE ≈ 0 | No hay tensión topológica | NO |

**Tres tests:**
1. **¿Existe ΔE no nula?** Comparar E₀(T³) con E₀(R³) al mismo N
2. **¿Cómo escala ΔE con V?** Barrer en N = 343, 729, 1331, 2197, 3375 y ajustar ley de potencia
3. **¿Cuál es la presión asociada?** w_top = P_top / (ρ_top · c²) usando teorema del virial sobre ΔE(V)

**Advertencia metodológica:** la diferencia E_T³ − E_R³ depende sensiblemente de cómo se trate el borde. Vamos a usar dos versiones de R³ (libre y con "absorbing boundary") para chequear robustez.

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_topologia'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

## Funciones del cristal — versiones T³ (cerrado) y R³ (abierto)

In [ ]:
def grid_T3(N_target, jitter_fraction=0.10, seed=42):
    """Cristal en T3 (caja unidad con condiciones periódicas)."""
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def grid_R3(N_target, jitter_fraction=0.10, seed=42):
    """Cristal en cubo R3 abierto (sin condiciones periódicas).
    Mismo posicionamiento que T3 pero sin wrap."""
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = np.clip(points, 0, 1)  # Sin wrap, solo clip
    return points, N_actual

def distance_matrix_T3(points, L=1.0):
    """Distancias con condiciones periódicas."""
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def distance_matrix_R3(points):
    """Distancias euclidianas sin wrap."""
    diff = points[:, None, :] - points[None, :, :]
    return np.linalg.norm(diff, axis=-1)

def build_laplacian(D_mat, k_neighbors=30, alpha=2.0):
    N = len(D_mat)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                rows.append(i); cols.append(j); vals.append(1.0/d**alpha)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W

print('Funciones listas')

## Test 1 — ¿Existe ΔE no nula entre T³ y R³ al mismo N?

Comparamos la energía de punto cero E₀ = Σ ½ω_n sobre los modos del cristal en ambas topologías, mismo tamaño físico, misma siembra de jitter.

In [ ]:
def calcular_E0(L_op, n_eigs=400):
    """Calcula E_0 = Σ½ω_n sobre los primeros n_eigs autovalores.
    Excluye modos cero (boundary modes en R³)."""
    N = L_op.shape[0]
    # Si pedimos demasiados, usar densa
    if n_eigs >= N - 1:
        from scipy.linalg import eigh
        eigs = eigh(L_op.toarray(), eigvals_only=True)
    else:
        eigs, _ = eigsh(L_op, k=n_eigs, which='SM', tol=1e-8)
    eigs = np.sort(eigs)
    eigs_nz = eigs[eigs > 1e-3]
    omegas = np.sqrt(eigs_nz)
    return np.sum(0.5 * omegas), len(eigs_nz), omegas

# Test para N = 1331 (referencia) — espectro completo
N_TEST = 1331
K_NEIGHBORS = 30
JITTER = 0.10

from scipy.linalg import eigh

print(f'Test 1: comparación T³ vs R³ con N = {N_TEST}\n')

# T3
points_T3, N_T3 = grid_T3(N_TEST, JITTER, seed=42)
D_T3 = distance_matrix_T3(points_T3, L=1.0)
L_T3 = build_laplacian(D_T3, K_NEIGHBORS, alpha=2.0)
t0 = time.time()
eigs_T3 = eigh(L_T3.toarray(), eigvals_only=True)
eigs_T3_nz = eigs_T3[eigs_T3 > 1e-3]
omegas_T3 = np.sqrt(eigs_T3_nz)
E0_T3 = np.sum(0.5 * omegas_T3)
n_modes_T3 = len(omegas_T3)
print(f'T³: E_0 = {E0_T3:.4f}  ({n_modes_T3} modos, {time.time()-t0:.1f}s)')
print(f'    ω: [{omegas_T3.min():.3f}, {omegas_T3.max():.3f}]')

# R3
points_R3, N_R3 = grid_R3(N_TEST, JITTER, seed=42)
D_R3 = distance_matrix_R3(points_R3)
L_R3 = build_laplacian(D_R3, K_NEIGHBORS, alpha=2.0)
t0 = time.time()
eigs_R3 = eigh(L_R3.toarray(), eigvals_only=True)
eigs_R3_nz = eigs_R3[eigs_R3 > 1e-3]
omegas_R3 = np.sqrt(eigs_R3_nz)
E0_R3 = np.sum(0.5 * omegas_R3)
n_modes_R3 = len(omegas_R3)
print(f'R³: E_0 = {E0_R3:.4f}  ({n_modes_R3} modos, {time.time()-t0:.1f}s)')
print(f'    ω: [{omegas_R3.min():.3f}, {omegas_R3.max():.3f}]')

Delta_E = E0_T3 - E0_R3
print(f'\nΔE_topológica = E_T³ − E_R³ = {Delta_E:+.4f}')
print(f'  Razón ΔE/E_T³ = {Delta_E/E0_T3*100:+.2f}%')

if abs(Delta_E)/E0_T3 < 0.01:
    print('  → ΔE pequeña: la topología no induce contribución residual significativa')
elif Delta_E > 0:
    print('  → T³ tiene MÁS energía que R³: tensión topológica residual POSITIVA')
else:
    print('  → R³ tiene MÁS energía que T³: efecto de borde domina')

## Test 2 — ¿Cómo escala ΔE con el volumen?

Repetimos el cálculo a varios N y ajustamos ΔE = A · V^β. El exponente β decide la naturaleza del efecto:

- β = 1: ρ_top constante → **Λ verdadera**
- β = 2/3: efecto de borde
- β = 1/3: efecto lineal
- β = -1: Casimir típico (se diluye)

In [ ]:
# Barrido en N
Ns_target = [343, 729, 1331, 2197, 3375]  # mismos que SIM 15
results = []

print(f'Test 2: escalamiento ΔE(V) en {len(Ns_target)} densidades\n')
print(f'{"N":>6} {"E_T3":>12} {"E_R3":>12} {"ΔE":>12} {"ΔE/N":>12}')
print('-'*70)

# IMPORTANTE METODOLÓGICAMENTE: para comparar T³ vs R³ honestamente,
# usamos el ESPECTRO COMPLETO (todos los autovalores) en cada caso.
# Esto evita el sesgo de "truncar a 400 modos" que falsea la comparación.
for Nt in Ns_target:
    t0 = time.time()
    
    # T3 — espectro completo
    points_T3, N = grid_T3(Nt, JITTER, seed=42)
    D = distance_matrix_T3(points_T3, L=1.0)
    L_op = build_laplacian(D, K_NEIGHBORS, alpha=2.0)
    from scipy.linalg import eigh
    eigs_T3 = eigh(L_op.toarray(), eigvals_only=True)
    eigs_T3_nz = eigs_T3[eigs_T3 > 1e-3]
    omegas_T3 = np.sqrt(eigs_T3_nz)
    E_T3 = np.sum(0.5 * omegas_T3)
    n_modes_T3 = len(omegas_T3)
    
    # R3 — espectro completo
    points_R3, _ = grid_R3(Nt, JITTER, seed=42)
    D = distance_matrix_R3(points_R3)
    L_op = build_laplacian(D, K_NEIGHBORS, alpha=2.0)
    eigs_R3 = eigh(L_op.toarray(), eigvals_only=True)
    eigs_R3_nz = eigs_R3[eigs_R3 > 1e-3]
    omegas_R3 = np.sqrt(eigs_R3_nz)
    E_R3 = np.sum(0.5 * omegas_R3)
    n_modes_R3 = len(omegas_R3)
    
    # Volumen físico (fijamos L=1 para todos los cubos)
    # En realidad lo que cambia con N es la DENSIDAD
    # Para evaluar como función de V "físico", asumimos espaciado a = 1/n_side fijo
    # entonces V_physical = (n_side · a)^3 con a fijo = 1
    n_side = round(Nt**(1/3))
    V_phys = float(n_side)**3  # Volumen en unidades de a^3
    
    Delta_E = E_T3 - E_R3
    
    results.append({
        'N': N,
        'V_phys': V_phys,
        'E_T3': E_T3,
        'E_R3': E_R3,
        'Delta_E': Delta_E,
        'rho_top': Delta_E / V_phys,
        'time': time.time() - t0
    })
    
    r = results[-1]
    print(f'{r["N"]:>6} {r["E_T3"]:>12.2f} {r["E_R3"]:>12.2f} '
          f'{r["Delta_E"]:>+12.4f} {r["Delta_E"]/r["N"]:>+12.5f}  '
          f'(t={r["time"]:.0f}s)')

In [ ]:
# Ajuste de ley de potencia: |ΔE| ∝ V^β
Vs = np.array([r['V_phys'] for r in results])
Ns = np.array([r['N'] for r in results])
DEs = np.array([r['Delta_E'] for r in results])

# Necesitamos |ΔE| > 0 para log
abs_DEs = np.abs(DEs)
if (abs_DEs > 0).all():
    log_V = np.log10(Vs)
    log_DE = np.log10(abs_DEs)
    coefs = np.polyfit(log_V, log_DE, 1)
    beta_V = coefs[0]
    A_V = 10**coefs[1]
    
    # También ajuste vs N
    log_N = np.log10(Ns)
    coefs_N = np.polyfit(log_N, log_DE, 1)
    beta_N = coefs_N[0]
    
    print(f'Ajustes de ley de potencia:')
    print(f'  |ΔE| ∝ V^β,  β = {beta_V:+.3f}')
    print(f'  |ΔE| ∝ N^γ,  γ = {beta_N:+.3f}')
    print()
    print(f'Interpretación:')
    print(f'  β = +1.00 ± 0.10: Λ verdadera (ρ_top constante)')
    print(f'  β = +0.67 ± 0.10: efecto de borde dominante')
    print(f'  β = +0.33 ± 0.10: efecto lineal')
    print(f'  β ≈ 0:            efecto puntual (sub-extensivo)')
    print(f'  β = -1:            Casimir típico (se diluye)')
    print()
    
    if abs(beta_V - 1.0) < 0.15:
        print(f'  → β ≈ {beta_V:.2f}: COMPATIBLE CON Λ ✓')
    elif abs(beta_V - 2/3) < 0.10:
        print(f'  → β ≈ {beta_V:.2f}: efecto de borde, NO es Λ')
    elif beta_V < 0:
        print(f'  → β ≈ {beta_V:.2f}: se diluye con la expansión, NO es Λ')
    else:
        print(f'  → β ≈ {beta_V:.2f}: comportamiento intermedio, requiere análisis adicional')
else:
    print('Algunos ΔE son cero - no se puede ajustar potencia')
    beta_V = None
    A_V = None

In [ ]:
# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: E_T3 vs E_R3 vs V
ax = axes[0]
E_T3s = np.array([r['E_T3'] for r in results])
E_R3s = np.array([r['E_R3'] for r in results])
ax.loglog(Ns, E_T3s, 'o-', markersize=12, lw=2, color='steelblue', label='E_0(T³)')
ax.loglog(Ns, E_R3s, 's-', markersize=12, lw=2, color='crimson', label='E_0(R³)')
ax.set_xlabel('N (número de nodos)', fontsize=12)
ax.set_ylabel('E_0 (energía punto cero)', fontsize=12)
ax.set_title('Energía punto cero: T³ cerrado vs R³ abierto', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

# Panel 2: ΔE vs V con ajuste
ax = axes[1]
if beta_V is not None:
    # Datos
    color_pts = 'green' if all(d > 0 for d in DEs) else ('red' if all(d < 0 for d in DEs) else 'gray')
    ax.loglog(Ns, abs_DEs, 'o', markersize=14, color=color_pts, label=f'|ΔE| medido')
    
    # Ajuste
    Ns_smooth = np.logspace(np.log10(Ns.min()*0.9), np.log10(Ns.max()*1.1), 50)
    Vs_smooth = Ns_smooth  # asumimos V ∝ N
    DE_fit = A_V * Vs_smooth**beta_V
    ax.loglog(Ns_smooth, DE_fit, '--', color='black', lw=2,
              label=f'Ajuste β = {beta_V:+.2f}')
    
    # Líneas de referencia
    DE_ref_lambda = abs_DEs[2] * (Ns_smooth/Ns[2])**1.0
    ax.loglog(Ns_smooth, DE_ref_lambda, ':', color='blue', alpha=0.5, label='β=+1 (Λ)')
    DE_ref_borde = abs_DEs[2] * (Ns_smooth/Ns[2])**(2/3)
    ax.loglog(Ns_smooth, DE_ref_borde, ':', color='orange', alpha=0.5, label='β=+2/3 (borde)')
    
    # Signo en cada punto
    for Ni, Di in zip(Ns, DEs):
        sign = '+' if Di > 0 else '-'
        ax.annotate(sign, (Ni, abs(Di)), fontsize=14, ha='center', va='bottom',
                   color='green' if Di > 0 else 'red')

ax.set_xlabel('N (volumen)', fontsize=12)
ax.set_ylabel('|ΔE| = |E_T³ − E_R³|', fontsize=12)
ax.set_title('Escalamiento de la diferencia topológica', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_plot('41_escalamiento_topologico')
plt.show()

## Test 3 — ¿Cuál es la presión de la diferencia topológica?

Calculamos w_top = P_top / (ρ_top · c²) usando el teorema del virial cuántico aplicado a ΔE(V).

Esto requiere repetir SIM 16 pero rastreando ΔE en lugar de E total. Variamos el volumen efectivo (cinco valores de ε) y medimos cómo cambia ΔE.

In [ ]:
# Para el test 3 necesitamos cambiar el volumen físico, no la cantidad de nodos
# Esto significa: mismo N=1331 pero con caja de tamaño L=(1+ε)

N_FIX = 1331
epsilons = [-0.02, -0.01, 0.0, 0.01, 0.02]

print('Test 3: cómo cambia ΔE con V físico (N fijo, caja escalada)\n')
print(f'{"ε":>6} {"L":>8} {"V":>8} {"E_T3":>12} {"E_R3":>12} {"ΔE":>12}')
print('-'*70)

data_t3 = []
for eps in epsilons:
    L_box = 1.0 + eps
    V_box = L_box**3
    
    # T3 — caja de lado L con condiciones periódicas, espectro completo
    points_T3, _ = grid_T3(N_FIX, JITTER, seed=42)
    points_T3_scaled = points_T3 * L_box
    D_T3 = distance_matrix_T3(points_T3_scaled, L=L_box)
    L_op_T3 = build_laplacian(D_T3, K_NEIGHBORS, alpha=2.0)
    eigs = eigh(L_op_T3.toarray(), eigvals_only=True)
    eigs_nz = eigs[eigs > 1e-3]
    E_T3 = np.sum(0.5 * np.sqrt(eigs_nz))
    
    # R3 — mismo cubo pero abierto, espectro completo
    points_R3, _ = grid_R3(N_FIX, JITTER, seed=42)
    points_R3_scaled = points_R3 * L_box
    D_R3 = distance_matrix_R3(points_R3_scaled)
    L_op_R3 = build_laplacian(D_R3, K_NEIGHBORS, alpha=2.0)
    eigs = eigh(L_op_R3.toarray(), eigvals_only=True)
    eigs_nz = eigs[eigs > 1e-3]
    E_R3 = np.sum(0.5 * np.sqrt(eigs_nz))
    
    DE = E_T3 - E_R3
    data_t3.append({'eps': eps, 'L': L_box, 'V': V_box, 
                     'E_T3': E_T3, 'E_R3': E_R3, 'Delta_E': DE})
    
    print(f'{eps:+.3f} {L_box:>8.4f} {V_box:>8.4f} {E_T3:>12.4f} {E_R3:>12.4f} {DE:>+12.4f}')

In [ ]:
# Calcular presión topológica
Vs_t3 = np.array([d['V'] for d in data_t3])
DEs_t3 = np.array([d['Delta_E'] for d in data_t3])

idx_0 = list(epsilons).index(0.0)

# Diferencia finita centrada para ∂(ΔE)/∂V
d_DE_dV = (DEs_t3[idx_0+1] - DEs_t3[idx_0-1]) / (Vs_t3[idx_0+1] - Vs_t3[idx_0-1])
P_top = -d_DE_dV

rho_top = DEs_t3[idx_0] / Vs_t3[idx_0]

# Velocidad del sonido (de notebooks anteriores)
c_s = 2.186

if abs(rho_top) > 1e-6:
    w_top = P_top / (rho_top * c_s**2)
else:
    w_top = float('nan')

print('='*70)
print('RESULTADO TEST 3: presión topológica')
print('='*70)
print()
print(f'  ∂(ΔE)/∂V = {d_DE_dV:+.6f}')
print(f'  P_top    = {P_top:+.6f}')
print(f'  ρ_top    = {rho_top:+.6f}')
print(f'  c_s²     = {c_s**2:.4f}')
print(f'  w_top    = {w_top:+.4f}')
print()
print(f'Comparación:')
print(f'  w cristal homogéneo (SIM 16) = +0.07')
print(f'  w defecto blando    (SIM 17) = +0.07')
print(f'  w_top topológico    (este)   = {w_top:+.4f}')
print(f'  w para ser Λ                  = -1.00')
print()

if abs(w_top + 1) < 0.2:
    print('  → ✓✓ COMPATIBLE CON Λ — la tensión topológica se comporta como energía oscura')
elif abs(w_top - 0.07) < 0.05:
    print('  → w_top idéntico al cristal/defectos: NO hay efecto distintivo de topología')
elif abs(w_top) > 0.5 and w_top < 0:
    print('  → w negativo grande pero no -1: contribución parcial a aceleración')
elif w_top > 0.2:
    print('  → w positivo: la diferencia topológica se comporta como materia/radiación')
else:
    print(f'  → comportamiento intermedio (w = {w_top:+.2f})')

In [ ]:
# Plot final con los tres tests
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test 3 visual
ax = axes[0]
ax.plot(Vs_t3, DEs_t3, 'o-', markersize=12, lw=2, color='purple')
ax.set_xlabel('V (volumen)', fontsize=12)
ax.set_ylabel('ΔE = E_T³ − E_R³', fontsize=12)
ax.set_title('ΔE(V): pendiente determina presión topológica', fontsize=12)
ax.grid(alpha=0.3)
ax.axhline(0, color='gray', lw=0.5)

# w_top vs benchmarks
ax = axes[1]
categories = ['Λ\n(w=-1)', 'Frontera\n(w=-1/3)', 'Polvo\n(w=0)', 
              'Cristal\n(w=+0.07)', 'Defecto\n(w=+0.07)',
              f'TOPOLOGÍA\n(w={w_top:+.2f})']
values = [-1, -1/3, 0, 0.07, 0.07, w_top]
colors = ['green', 'purple', 'gray', 'steelblue', 'orange', 'crimson']
ax.barh(categories, values, color=colors, alpha=0.7)
ax.axvline(0, color='black', lw=1)
ax.axvline(-1, color='green', linestyle='--', alpha=0.5, label='Λ requerido')
ax.set_xlabel('Parámetro de ecuación de estado w', fontsize=12)
ax.set_title('w_top topológico vs benchmarks', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
save_plot('42_w_topologico')
plt.show()

In [ ]:
# Síntesis final
print('='*75)
print('SÍNTESIS — Tensión topológica del vacío T³ vs R³')
print('='*75)
print()
print('Los tres tests:')
print()
print(f'TEST 1 — ¿Existe ΔE = E_T³ − E_R³ no nula?')
print(f'  ΔE(N=1331) = {results[2]["Delta_E"]:+.4f}')
if abs(results[2]['Delta_E']) > 0.5:
    print(f'  → SÍ: hay diferencia significativa entre topologías')
else:
    print(f'  → ΔE pequeña; topología no induce contribución dramática')
print()

if beta_V is not None:
    print(f'TEST 2 — ¿Cómo escala ΔE con V?')
    print(f'  Ajuste: |ΔE| ∝ V^{beta_V:+.2f}')
    if abs(beta_V - 1.0) < 0.15:
        print(f'  → β ≈ 1: ρ_top APROXIMADAMENTE CONSTANTE (candidato a Λ)')
    elif abs(beta_V - 2/3) < 0.10:
        print(f'  → β ≈ 2/3: efecto de BORDE (R³) dominante')
    elif beta_V < 0:
        print(f'  → β < 0: se DILUYE con expansión (Casimir, no Λ)')
    else:
        print(f'  → β = {beta_V:.2f}: comportamiento intermedio')
    print()

print(f'TEST 3 — ¿w cosmológico de la diferencia topológica?')
print(f'  w_top = {w_top:+.4f}')
if abs(w_top + 1) < 0.2:
    print(f'  → ✓✓ w ≈ -1: COMPATIBLE CON Λ')
elif abs(w_top - 0.07) < 0.05:
    print(f'  → w idéntico a cristal/defectos: misma fenomenología, no novedad')
else:
    print(f'  → w intermedio: requiere análisis adicional')
print()

print('VEREDICTO PARA LA HIPÓTESIS DE BRUSCHI:')
print()
tests_pass = []
if abs(results[2]['Delta_E']) > 0.5:
    tests_pass.append('1')
if beta_V is not None and abs(beta_V - 1.0) < 0.15:
    tests_pass.append('2')
if abs(w_top + 1) < 0.2:
    tests_pass.append('3')

if len(tests_pass) == 3:
    print('  ✓✓✓ LOS TRES TESTS PASAN')
    print('  La tensión topológica del cristal cerrado T³ se comporta cosmológicamente')
    print('  como constante cosmológica. Esto sería un resultado MUY FUERTE para DEE.')
elif len(tests_pass) >= 1:
    print(f'  Pasan los tests: {", ".join(tests_pass)} de 3')
    print('  Hay efecto topológico medible pero no idéntico a Λ pura.')
    print('  Posible: contribución parcial, mecanismo a refinar.')
else:
    print('  Ningún test pasa los criterios para Λ.')
    print('  La hipótesis de tensión topológica NO se verifica en este cálculo.')
    print('  Esto NO descarta la idea — sólo dice que en su forma más simple no funciona.')

In [ ]:
import shutil
shutil.make_archive('plots_topologia', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_topologia.zip')

try:
    from google.colab import files
    files.download('plots_topologia.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass